# llm-memlab Kaggle Quickstart

Notebook นี้แบ่งเป็น 5 ส่วนชัด ๆ: **Setup** ติดตั้ง lib, **Load Model** โหลดโมเดลจาก Hugging Face, **Debug Runtime** ตรวจ backend/GPU และ local fixtures, **Optimize/Benchmark** รัน memory-first + serving benchmark, และ **View Report** เปิด HTML dashboard.

## 1. Setup

Clone project และติดตั้ง package. Kaggle อาจต้องเปิด Internet ใน notebook settings ก่อน cell นี้.

In [ ]:
!git clone https://github.com/thhanabun/LLMOptimization.git
%cd LLMOptimization
!python -m pip install -e .[torch,transformers]

## 2. Configure Paths

Kaggle `/kaggle/input` เป็น read-only ดังนั้นโหลด Hugging Face model ไปที่ `/kaggle/working/hf_models` และเขียน reports ไปที่ `/kaggle/working/llm_memlab_outputs`.

In [ ]:
import os
os.environ.setdefault("LLM_MEMLAB_MODEL_ROOT", "/kaggle/working/hf_models")
os.environ.setdefault("LLM_MEMLAB_OUT_DIR", "/kaggle/working/llm_memlab_outputs")
os.environ.setdefault("LLM_MEMLAB_DEVICE", "auto")
os.environ.setdefault("LLM_MEMLAB_DTYPE", "auto")

## 3. Load Model From Hugging Face

โหลด model ลง `/kaggle/working` ด้วย `snapshot_download` แล้วตั้ง `LLM_MEMLAB_MODEL` ให้ workflow ใช้ต่อ.

- ค่า default เป็น tiny random Llama เพื่อ smoke test เร็ว ๆ
- เปลี่ยน `HF_MODEL_ID` เป็น model จริงได้ เช่น `TinyLlama/TinyLlama-1.1B-Chat-v1.0`
- ถ้าเป็น gated/private model ให้สร้าง Kaggle Secret ชื่อ `HF_TOKEN` แล้ว uncomment block ด้านล่าง

In [ ]:
# Optional for gated/private models:
# from kaggle_secrets import UserSecretsClient
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

from huggingface_hub import snapshot_download

HF_MODEL_ID = os.environ.get("HF_MODEL_ID", "hf-internal-testing/tiny-random-LlamaForCausalLM")
HF_TOKEN = os.environ.get("HF_TOKEN")
local_dir = os.path.join(os.environ["LLM_MEMLAB_MODEL_ROOT"], HF_MODEL_ID.replace("/", "__"))

model_path = snapshot_download(repo_id=HF_MODEL_ID, local_dir=local_dir, token=HF_TOKEN)
os.environ["LLM_MEMLAB_MODEL"] = model_path
print("Downloaded model to:", model_path)

## 4. Debug Runtime

ตรวจ backend/GPU/vLLM fallback และ scan local model folder. ส่วนนี้ช่วย debug ว่า environment พร้อมแค่ไหนก่อน optimize.

In [ ]:
!python -m llm_memlab backend-demo
!python -m llm_memlab estimate --preset 7b-like --seq 2048 --batch 1 --training inference
!python -m llm_memlab local-model-harness --root "$LLM_MEMLAB_MODEL_ROOT" --json-out /kaggle/working/local_model_fixtures.json

## 5. Optimize / Benchmark

รัน workflow หลัก: `memory-first-hf-bench`, `serving-bench`, และ export benchmark dashboard. ถ้า backend ใดไม่พร้อม จะถูกบันทึกเป็น fallback reason.

In [ ]:
!python examples/cloud_smoke.py --tokens 1

## 6. View Report

เปิด HTML dashboard ใน notebook. ไฟล์ทั้งหมดอยู่ใต้ `/kaggle/working/llm_memlab_outputs` เพื่อ download ได้จาก output panel.

In [ ]:
from IPython.display import HTML, display
dashboard = os.path.join(os.environ["LLM_MEMLAB_OUT_DIR"], "cloud_dashboard.html")
if os.path.exists(dashboard):
    display(HTML(open(dashboard, encoding="utf-8").read()))